# Flight Fare Prediction

**Tabular Regression Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `fare`

## 2 · Project Overview

We predict **flight ticket fares** based on airline, route (source/destination), distance, booking advance, departure time, number of stops, and seat class.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Encode categorical features for tree-based and linear models.
3. Build a baseline Linear Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with RMSE, MAE, R², and residual analysis.

## 4 · Problem Statement

Given an airline, route, distance, stops, booking lead time, departure hour, weekend flag, and seat class, predict the ticket fare.

## 5 · Why This Project Matters

- **Fare prediction** helps travelers find the best booking time.
- Airlines use dynamic pricing — understanding drivers is key.
- Travel agencies can optimize recommendations.
- Demonstrates regression with multiplicative pricing factors.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 9,000 |
| **Features** | airline, source_city, dest_city, distance_miles, duration_hours, stops, days_before_departure, departure_hour, is_weekend_departure, seat_class |
| **Target** | fare (continuous, USD) |
| **Range** | ~$30 – $5,000 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "fare"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

Synthetic dataset: 9,000 flight bookings with route and booking features.

In [ ]:
np.random.seed(SEED)
n = 9000
airline = np.random.choice(["Budget Air", "National", "Premium Wings", "SkyFirst"], n,
                            p=[0.3, 0.3, 0.25, 0.15])
airline_base = {"Budget Air": 80, "National": 150, "Premium Wings": 250, "SkyFirst": 400}
base = np.array([airline_base[a] for a in airline], dtype=float)

source_city = np.random.choice(["NYC", "LAX", "CHI", "MIA", "DFW"], n)
dest_city = np.random.choice(["NYC", "LAX", "CHI", "MIA", "DFW"], n)
# Ensure source != dest
mask = source_city == dest_city
while mask.any():
    dest_city[mask] = np.random.choice(["NYC", "LAX", "CHI", "MIA", "DFW"], mask.sum())
    mask = source_city == dest_city

distance_miles = np.random.randint(300, 3000, n)
duration_hours = np.round(distance_miles / np.random.uniform(400, 550, n), 1)
stops = np.random.choice([0, 1, 2], n, p=[0.5, 0.35, 0.15])
days_before_departure = np.random.randint(1, 180, n)
departure_hour = np.random.randint(5, 23, n)
is_weekend_departure = np.random.binomial(1, 0.3, n)
seat_class = np.random.choice(["Economy", "Business", "First"], n, p=[0.65, 0.25, 0.1])
class_mult = np.where(seat_class == "First", 3.5, np.where(seat_class == "Business", 2.0, 1.0))

# Fare model
advance_discount = np.where(days_before_departure > 60, 0.7,
                   np.where(days_before_departure > 21, 0.85,
                   np.where(days_before_departure > 7, 1.0, 1.3)))
stop_discount = np.where(stops == 0, 1.15, np.where(stops == 1, 1.0, 0.85))
peak_hour = np.where((departure_hour >= 7) & (departure_hour <= 9), 1.1,
            np.where((departure_hour >= 17) & (departure_hour <= 19), 1.1, 1.0))

fare = base + 0.08 * distance_miles
fare = fare * class_mult * advance_discount * stop_discount * peak_hour
fare = fare * (1 + 0.1 * is_weekend_departure)
fare = np.round(fare + np.random.normal(0, 20, n), 2).clip(30, 5000)

df = pd.DataFrame({"airline": airline, "source_city": source_city, "dest_city": dest_city,
                    "distance_miles": distance_miles, "duration_hours": duration_hours,
                    "stops": stops, "days_before_departure": days_before_departure,
                    "departure_hour": departure_hour, "is_weekend_departure": is_weekend_departure,
                    "seat_class": seat_class, "fare": fare})
print(f"Dataset shape: {df.shape}")
print(f"Target stats:\n{df['fare'].describe()}")
df.head()

## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")
print(f"\nTarget stats:\n{df[TARGET].describe()}")

## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0][0].scatter(df["distance_miles"], df[TARGET], alpha=0.2, s=10)
axes[0][0].set_xlabel("Distance"); axes[0][0].set_ylabel("Fare"); axes[0][0].set_title("Distance vs Fare")

axes[0][1].scatter(df["days_before_departure"], df[TARGET], alpha=0.2, s=10)
axes[0][1].set_xlabel("Days Before"); axes[0][1].set_ylabel("Fare"); axes[0][1].set_title("Advance Booking vs Fare")

df.boxplot(column=TARGET, by="airline", ax=axes[0][2])
axes[0][2].set_title("Fare by Airline")
axes[0][2].tick_params(axis="x", rotation=45)

df.boxplot(column=TARGET, by="seat_class", ax=axes[1][0])
axes[1][0].set_title("Fare by Seat Class")

df.boxplot(column=TARGET, by="stops", ax=axes[1][1])
axes[1][1].set_title("Fare by Stops")

corr = df.select_dtypes(include="number").corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1][2], square=True, cbar_kws={"shrink":0.8})
axes[1][2].set_title("Correlation")

plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `fare`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[TARGET], bins=30, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}")
axes[0].set_xlabel("Fare ($)")

df.groupby("seat_class")[TARGET].median().reindex(["Economy","Business","First"]).plot(
    kind="bar", ax=axes[1], color="steelblue", edgecolor="black")
axes[1].set_title("Median Fare by Seat Class")

plt.tight_layout()
plt.show()
print(f"Range: ${df[TARGET].min():,.0f} to ${df[TARGET].max():,.0f}")
print(f"Mean: ${df[TARGET].mean():,.0f}, Median: ${df[TARGET].median():,.0f}")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target — Train mean: {y_train.mean():,.2f}, Test mean: {y_test.mean():,.2f}")

## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `airline`, `source_city`, `dest_city`, `seat_class` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Outliers**: First-class fares create right skew — keep them.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [ ]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["fare_per_mile_proxy"] = X_train["distance_miles"] / (X_train["duration_hours"] + 0.1)
X_test["fare_per_mile_proxy"] = X_test["distance_miles"] / (X_test["duration_hours"] + 0.1)

X_train["is_last_minute"] = (X_train["days_before_departure"] < 7).astype(int)
X_test["is_last_minute"] = (X_test["days_before_departure"] < 7).astype(int)

X_train["is_peak_hour"] = ((X_train["departure_hour"] >= 7) & (X_train["departure_hour"] <= 9) |
                            (X_train["departure_hour"] >= 17) & (X_train["departure_hour"] <= 19)).astype(int)
X_test["is_peak_hour"] = ((X_test["departure_hour"] >= 7) & (X_test["departure_hour"] <= 9) |
                           (X_test["departure_hour"] >= 17) & (X_test["departure_hour"] <= 19)).astype(int)

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

## 18 · Baseline Model

Linear Regression as our baseline regressor.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Fit dozens of regressors in one call for a quick ranking.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Regressors:")
print(lazy_models.head(15).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [ ]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred_flaml):,.2f}")
print(f"R2  : {r2_score(y_test, y_pred_flaml):.4f}")

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Linear Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)

    axes[i].scatter(y_test, yp, alpha=0.6, s=40, edgecolors="black")
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_predicted_vs_actual.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.6, s=40, edgecolors="black")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

sorted_idx = np.argsort(best_pred)
axes[2].scatter(best_pred[sorted_idx], np.abs(residuals[sorted_idx]), alpha=0.6, s=40, edgecolors="black")
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")
print(f"  Median: {np.median(residuals):,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Seat class** is the strongest fare multiplier (First=3.5×, Business=2×).
- **Airline tier** sets the base price level.
- **Distance** has a linear component per mile.
- **Advance booking** provides significant discounts (>60 days = 30% off).
- **Last-minute bookings** are 30% more expensive.
- **Direct flights** carry a ~15% premium over connections.

**Business takeaway:** Book 60+ days early for best fares. Direct flights on budget airlines offer the best value.

## 26 · Limitations

1. Synthetic pricing model — real fares use yield management.
2. No demand/load factor data.
3. Missing competitor pricing.
4. No seasonal or holiday variation.
5. Simplified route network.

## 27 · How to Improve This Project

1. Use real flight booking data (scraping or APIs).
2. Add demand indicators (search volume, seat availability).
3. Include seasonal and holiday features.
4. Model dynamic pricing with temporal sequences.
5. Add baggage and ancillary revenue.

## 28 · Production Considerations

- Deploy as a fare prediction/recommendation API.
- Show fare trends and optimal booking windows.
- Monitor for airline pricing strategy changes.
- Update weekly with new fare data.
- Output fare confidence intervals.

## 29 · Common Mistakes

1. Using simple averages instead of modeling multiplicative pricing.
2. Not accounting for seat class (the biggest driver).
3. Ignoring advance booking discounts.
4. Treating all airlines equally.
5. Not log-transforming right-skewed fares.

## 30 · Mini Challenge / Exercises

1. Log-transform `fare` — does R² improve for Linear Regression?
2. Remove `seat_class` — how much does RMSE increase?
3. Plot fare vs. days_before_departure — identify discount tiers.
4. Create `cost_per_mile = fare / distance_miles` and analyze by airline.
5. Build separate models for Economy vs. Business/First.

## 31 · Final Summary / Key Takeaways

1. **Seat class** is the strongest fare multiplier.
2. **Airline tier** sets the base price level.
3. **Distance** adds a linear per-mile component.
4. **Advance booking** (60+ days) saves ~30%.
5. **Direct flights** command a premium.
6. **Dynamic pricing** in production requires real-time demand data.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))